In [7]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain.tools import BaseTool, StructuredTool, tool
from langchain_openai import ChatOpenAI, OpenAI
from langchain import hub
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor

from langchain_community.llms import LlamaCpp
from langchain_community.chat_models import ChatLlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [4]:
n_gpu_layers = 1  # The number of layers to put on the GPU. The rest will be on the CPU. If you don't know how many layers there are, you can use -1 to move all to GPU.
n_batch = 512  # Should be between 1 and n_ctx, consider the amount of RAM of your Apple Silicon Chip.
# Make sure the model path is correct for your system!
llm = ChatLlamaCpp(
    model_path="../models/Llama-3-Groq-8B-Tool-Use-Q4_K_M.gguf",
    # model_path="../models/Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf",
    # model_path="../models/Hermes-2-Pro-Llama-3-8B-Q4_K_M.gguf",
    n_gpu_layers=n_gpu_layers,
    n_batch=n_batch,
    f16_kv=True,  # MUST set to True, otherwise you will run into problem after a couple of calls
    verbose=True,  # Verbose is required to pass to the callback manager
)

llama_model_loader: loaded meta data with 27 key-value pairs and 291 tensors from ../models/Llama-3-Groq-8B-Tool-Use-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = Llama-3-Groq-8B-Tool-Use
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.attention.head_count u32              = 32
llama_model_loader: - kv   7:              llama.

In [15]:
@tool
def offer(item: str, price: float):
  """
  Offer an item for sale at a given price.
  """
  print(f"OFFERING {item} FOR {price}")
  pass

@tool
def sell(item: str, price: float):
  """
  Sell an item at a given price. This can only be done after the shopkeeper has accepted an offer
  """
  print(f"SELLING {item} FOR {price}")
  pass

@tool
def rescind_offer(item: str):
  """
  Rescind an offer for an item. This means the seller is no longer willing to sell the item.
  """
  print(f"RESCINDING OFFER FOR {item}")
  pass

@tool
def leave_shop():
  """
  Leave the shop. This means the seller is no longer interested in selling anything.
  """
  print("LEAVING SHOP")
  pass

In [16]:
tools = [offer, sell, rescind_offer, leave_shop]
prompt_template = ChatPromptTemplate([
    ("system", """You are Erik Stoneforge. Erik Stoneforge is a seasoned adventurer in his mid-thirties, known for his sharp eye and shrewdness in negotiations. He steps into the pawn shop with a worn leather satchel containing carefully selected goods from his recent journey.
Inside are a finely crafted silver dagger etched with mysterious runes, which Erik believes holds more value to collectors of rare weaponry than to any common buyer. He also carries a cracked mana crystal, still faintly glowing, knowing it’s imperfect but hoping to fetch a decent price from someone seeking magical components. Lastly, an ancient bronze amulet adorned with emeralds catches the eye, and Erik is keen to emphasize the historical significance of the piece to drive up its value.
Erik prefers to haggle based on the uniqueness or rarity of each item, especially when he senses a merchant might undervalue magical or historical goods. He’s patient but firm in his negotiations, and while he’s willing to compromise on the mana crystal, he’s prepared to walk away if he doesn’t get a good offer for the amulet or the dagger.

You are here to haggle with the shopkeeper.
"""),
    MessagesPlaceholder("msgs")
])

model_with_tools = llm.bind_tools(tools)
query = "Hello Erik! I see you have some interesting items for sale. What can I do for you today?"
messages = [HumanMessage(query)]
prompt = prompt_template.invoke({"msgs": messages})

ai_msg = model_with_tools.invoke(prompt)
messages.append(ai_msg)
messages

Llama.generate: 2 prefix-match hit, remaining 270 prompt tokens to eval
llama_perf_context_print:        load time =   18548.68 ms
llama_perf_context_print: prompt eval time =       0.00 ms /   270 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /    43 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   32857.03 ms /   313 tokens


[HumanMessage(content='Hello Erik! I see you have some interesting items for sale. What can I do for you today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Good day! Yes, I have some rare and valuable items. For starters, I have a finely crafted silver dagger etched with mysterious runes. It's quite unique and I believe it could fetch a good price.", additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-4c046510-b022-454c-9e05-d6c33f75be55-0')]

In [12]:
messages

AIMessage(content="Good day! I'm looking to sell these items and possibly purchase something in return. Do you have any items that might match what I’m seeking?", additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-38bbdda8-c89c-4c02-8944-27fa302ea9ca-0')

In [14]:
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
messages

Llama.generate: 62 prefix-match hit, remaining 5 prompt tokens to eval
llama_perf_context_print:        load time =    8210.47 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     5 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /    34 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   12173.93 ms /    39 tokens


[HumanMessage(content='I need to get to MPK 17 as fast as possible. Please get me there as fast as possible.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm sorry, but I don't have the capability to physically transport you to any location. Is there anything else I can assist you with?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 33, 'completion_tokens': 29, 'total_tokens': 62}, 'finish_reason': 'stop', 'logprobs': None}, id='run-2cf6b44b-ed0b-4d30-8269-463f0e4fa79f-0'),
 AIMessage(content="If you need directions or assistance with planning your trip, I can certainly help. Just let me know the details of where you're going and any specific requirements you have.", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 67, 'completion_tokens': 34, 'total_tokens': 101}, 'finish_reason': 'stop', 'logprobs': None}, id='run-24bac57d-cc21-496d-bd0a-0dab6c454db2-0')]